# Reproducible Testing

Create consistent test environments with clean memory branches for each test scenario.

**Problem**: Agent behavior varies between test runs due to memory state differences, making it hard to isolate specific memory scenarios.

In [ ]:
import asyncio
from memoir.core.memory import ProllyTreeMemoryStoreManager
from memoir.classifier.intelligent import IntelligentClassifier
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(model="gpt-4", temperature=0)
classifier = IntelligentClassifier(llm=llm)
memory = ProllyTreeMemoryStoreManager(classifier=classifier)

In [ ]:
# Create baseline test state
baseline_commit = await memory.store_memory(
    "User is a software engineer",
    metadata={"message": "Test baseline"}
)

await memory.store_memory(
    "User knows Python and JavaScript",
    metadata={"message": "Test baseline skills"}
)

print(f"Baseline test state: {baseline_commit[:8]}")

In [ ]:
# Test Scenario 1: Beginner user
await memory.create_branch("test_beginner")
await memory.checkout("test_beginner")

await memory.store_memory(
    "User is new to programming, learning Python basics",
    metadata={"message": "Test scenario: beginner"}
)

beginner_results = await memory.search_memories("user programming experience")
print("Test 1 - Beginner scenario:")
print(f"Memory count: {len(beginner_results)}")
print(f"Experience level: {beginner_results[0].content}")

In [ ]:
# Test Scenario 2: Expert user (clean slate)
await memory.checkout("main")
await memory.create_branch("test_expert")
await memory.checkout("test_expert")

await memory.store_memory(
    "User is a senior engineer with 10 years experience",
    metadata={"message": "Test scenario: expert"}
)

expert_results = await memory.search_memories("user programming experience")
print("\nTest 2 - Expert scenario:")
print(f"Memory count: {len(expert_results)}")
print(f"Experience level: {expert_results[0].content}")

In [ ]:
# Test Scenario 3: Mixed experience (another clean slate)
await memory.checkout("main")
await memory.create_branch("test_mixed")
await memory.checkout("test_mixed")

await memory.store_memory(
    "User is expert in Python but beginner in JavaScript",
    metadata={"message": "Test scenario: mixed"}
)

mixed_results = await memory.search_memories("user programming experience")
print("\nTest 3 - Mixed experience scenario:")
print(f"Memory count: {len(mixed_results)}")
print(f"Experience level: {mixed_results[0].content}")

In [ ]:
# Verify test isolation
await memory.checkout("test_beginner")
beginner_check = await memory.search_memories("senior engineer")
print(f"\nIsolation check - Beginner branch has 'senior engineer': {len(beginner_check) > 0}")

await memory.checkout("test_expert")
expert_check = await memory.search_memories("new to programming")
print(f"Isolation check - Expert branch has 'new to programming': {len(expert_check) > 0}")

In [ ]:
# Rerun same test - should be identical
await memory.checkout("main")
await memory.delete_branch("test_beginner")
await memory.create_branch("test_beginner")
await memory.checkout("test_beginner")

await memory.store_memory(
    "User is new to programming, learning Python basics",
    metadata={"message": "Test scenario: beginner (rerun)"}
)

rerun_results = await memory.search_memories("user programming experience")
print("\nReproducibility check - Beginner test rerun:")
print(f"Memory count: {len(rerun_results)} (should be 3)")
print(f"Content matches: {rerun_results[0].content == beginner_results[0].content}")

In [ ]:
# Cleanup test branches
await memory.checkout("main")
test_branches = ["test_beginner", "test_expert", "test_mixed"]

for branch in test_branches:
    await memory.delete_branch(branch)
    print(f"Deleted test branch: {branch}")

print("\nAll test environments cleaned up, main branch preserved")